In [ ]:
print("Machine Fault Detection Project")
print("Jupyter Notebook is working!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

In [ ]:
df = pd.read_csv("machine_fault_project_dataset_5_parameters_random_ids.csv")

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
df.head()

In [ ]:
print("Fault types:")
print(df["Fault_Type"].value_counts())

print("\nSeverity levels:")
print(df["Severity"].value_counts())

In [ ]:
features = [
    "Temperature_C",
    "Vibration_RMS_g",
    "Current_RMS_A",
    "Acoustic_RMS_Pa",
    "RPM"
]

print("Selected sensor parameters:")
for feature in features:
    print(feature)

In [ ]:
X = df[features].to_numpy()

print("Feature matrix created successfully!")
print("Matrix shape:", X.shape)

In [ ]:
print("First 5 machine observations:")
print(X[:5])

## 3. Formation of the Sensor Feature Matrix

The industrial machine is monitored using five sensor parameters:

1. Temperature
2. Vibration
3. Motor Current
4. Acoustic Signal
5. RPM

Each machine observation is represented as a five-dimensional vector:

x = [Temperature, Vibration, Current, Acoustic, RPM]

A collection of 2100 observations forms the feature matrix X.

Therefore,

X ∈ R^(2100 × 5)

where each row represents one machine observation and each column
represents one sensor parameter.

This matrix is the foundation for the linear algebra operations
used in the subsequent stages of the project.

In [ ]:
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)

print("Mean of each sensor parameter:")
print(X_mean)

print("\nStandard deviation of each sensor parameter:")
print(X_std)

In [ ]:
X_standardized = (X - X_mean) / X_std

print("Standardized matrix shape:", X_standardized.shape)

In [ ]:
print("First 5 standardized observations:")
print(X_standardized[:5])

In [ ]:
print("Mean after standardization:")
print(X_standardized.mean(axis=0))

print("\nStandard deviation after standardization:")
print(X_standardized.std(axis=0))

In [ ]:
correlation_matrix = np.corrcoef(X_standardized, rowvar=False)

print("Correlation matrix:")
print(correlation_matrix)

In [ ]:
import sympy as sp

C = sp.Matrix(correlation_matrix)

rref_matrix, pivot_columns = C.rref()

print("RREF of the correlation matrix:")
print(rref_matrix)

print("\nPivot columns:")
print(pivot_columns)

In [ ]:
rank = len(pivot_columns)

print("Rank of the correlation matrix:", rank)
print("Number of sensor parameters:", len(features))

In [ ]:
sensor_vectors = X_standardized.T

print("Shape of sensor vectors:", sensor_vectors.shape)

In [ ]:
def gram_schmidt(vectors):
    orthogonal_vectors = []

    for v in vectors:
        u = v.copy()

        for q in orthogonal_vectors:
            projection = np.dot(v, q) / np.dot(q, q) * q
            u = u - projection

        orthogonal_vectors.append(u)

    return np.array(orthogonal_vectors)

In [ ]:
orthogonal_vectors = gram_schmidt(sensor_vectors)

print("Orthogonal basis shape:", orthogonal_vectors.shape)

In [ ]:
orthogonality_check = np.dot(
    orthogonal_vectors,
    orthogonal_vectors.T
)

print("Orthogonality check:")
print(np.round(orthogonality_check, 5))

In [ ]:
orthonormal_basis = []

for q in orthogonal_vectors:
    norm = np.linalg.norm(q)
    orthonormal_basis.append(q / norm)

orthonormal_basis = np.array(orthonormal_basis)

print("Orthonormal basis created successfully!")
print("Shape:", orthonormal_basis.shape)

In [ ]:
orthonormal_check = np.dot(
    orthonormal_basis,
    orthonormal_basis.T
)

print(np.round(orthonormal_check, 5))

In [ ]:
normal_data = df[df["Fault_Type"] == "Normal"]

X_normal = normal_data[features].to_numpy()

print("Number of normal observations:", X_normal.shape[0])
print("Normal data shape:", X_normal.shape)

In [ ]:
X_normal_standardized = (X_normal - X_mean) / X_std

print("Standardized normal data shape:",
      X_normal_standardized.shape)

In [ ]:
normal_vector_1 = X_normal_standardized[0]
normal_vector_2 = X_normal_standardized[1]

print("Normal vector 1:")
print(normal_vector_1)

print("\nNormal vector 2:")
print(normal_vector_2)

In [ ]:
normal_vectors = np.array([
    normal_vector_1,
    normal_vector_2
])

normal_orthogonal = gram_schmidt(normal_vectors)

print("Orthogonal normal vectors:")
print(normal_orthogonal)

In [ ]:
normal_basis = []

for v in normal_orthogonal:
    norm = np.linalg.norm(v)
    normal_basis.append(v / norm)

normal_basis = np.array(normal_basis)

print("Normal orthonormal basis:")
print(normal_basis)

In [ ]:
print("Basis verification:")
print(np.round(
    np.dot(normal_basis, normal_basis.T),
    5
))

In [ ]:
test_normal = X_normal_standardized[10]

print("Test normal machine observation:")
print(test_normal)

In [ ]:
normal_coordinates = normal_basis @ test_normal

normal_projection = normal_basis.T @ normal_coordinates

print("Projection coordinates:")
print(normal_coordinates)

print("\nProjected normal observation:")
print(normal_projection)

In [ ]:
normal_residual = test_normal - normal_projection

normal_error = np.linalg.norm(normal_residual)

print("Projection residual:")
print(normal_residual)

print("\nNormal projection error:", normal_error)

In [ ]:
fault_data = df[df["Fault_Type"] != "Normal"]

X_fault = fault_data[features].to_numpy()

X_fault_standardized = (X_fault - X_mean) / X_std

test_fault = X_fault_standardized[0]

print("Test faulty machine observation:")
print(test_fault)

In [ ]:
fault_coordinates = normal_basis @ test_fault

fault_projection = normal_basis.T @ fault_coordinates

fault_residual = test_fault - fault_projection

fault_error = np.linalg.norm(fault_residual)

print("Fault projection error:", fault_error)

In [ ]:
print("Normal machine projection error:", normal_error)
print("Faulty machine projection error:", fault_error)

if fault_error > normal_error:
    print("\nThe faulty machine is farther from the normal operating subspace.")
else:
    print("\nThis particular fault sample was not farther from the subspace.")

In [ ]:
# Four sensor parameters used to predict temperature

X_prediction = X_standardized[:, [1, 2, 3, 4]]

# Actual temperature
y_temperature = X_standardized[:, 0]

print("Prediction matrix shape:", X_prediction.shape)
print("Temperature vector shape:", y_temperature.shape)

In [ ]:
A = np.column_stack([
    np.ones(X_prediction.shape[0]),
    X_prediction
])

print("Design matrix shape:", A.shape)

In [ ]:
coefficients, residuals, rank_ls, singular_values = np.linalg.lstsq(
    A,
    y_temperature,
    rcond=None
)

print("Least-squares coefficients:")
print(coefficients)

print("\nRank of design matrix:", rank_ls)

In [ ]:
y_predicted = A @ coefficients

print("First 10 actual standardized temperatures:")
print(y_temperature[:10])

print("\nFirst 10 predicted standardized temperatures:")
print(y_predicted[:10])

In [ ]:
prediction_error = y_temperature - y_predicted

print("First 10 prediction errors:")
print(prediction_error[:10])

In [ ]:
mse = np.mean(prediction_error ** 2)

print("Mean Squared Error:", mse)

In [ ]:
rmse = np.sqrt(mse)

print("Root Mean Squared Error:", rmse)

In [ ]:
ss_total = np.sum((y_temperature - y_temperature.mean()) ** 2)
ss_residual = np.sum((y_temperature - y_predicted) ** 2)

r_squared = 1 - (ss_residual / ss_total)

print("R² score:", r_squared)

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_temperature,
    y_predicted,
    alpha=0.5
)

plt.xlabel("Actual Standardized Temperature")
plt.ylabel("Predicted Standardized Temperature")
plt.title("Least-Squares: Actual vs Predicted Temperature")

plt.axline((0, 0), slope=1)

plt.tight_layout()
plt.show()

## 6. Eigenvalues and Eigenvectors for Machine Operating Patterns

Eigenvalues and eigenvectors are used to identify the dominant patterns
present in the normal operating condition of the machine.

The covariance matrix of the normal machine observations is constructed.
Since the covariance matrix is symmetric, it can be decomposed using
eigenvalues and eigenvectors.

Large eigenvalues represent directions containing more variation in
the machine's normal operating behavior.

In [ ]:
covariance_matrix = np.cov(
    X_normal_standardized,
    rowvar=False
)

print("Covariance matrix:")
print(covariance_matrix)

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(
    covariance_matrix
)

print("Eigenvalues:")
print(eigenvalues)

print("\nEigenvectors:")
print(eigenvectors)

In [ ]:
idx = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Eigenvalues in descending order:")
print(eigenvalues)

In [ ]:
explained_variance_ratio = (
    eigenvalues / eigenvalues.sum()
)

cumulative_variance = np.cumsum(
    explained_variance_ratio
)

print("Explained variance ratio:")
print(explained_variance_ratio)

print("\nCumulative explained variance:")
print(cumulative_variance)

In [ ]:
for i in range(len(eigenvalues)):
    print(
        "Eigenvector", i + 1,
        ":",
        round(explained_variance_ratio[i] * 100, 2),
        "% variance"
    )

In [ ]:
# Select the first 2 eigenvectors
k = 2

Q_normal = eigenvectors[:, :k]

print("Normal operating basis:")
print(Q_normal)

print("\nBasis shape:", Q_normal.shape)

In [ ]:
basis_check = Q_normal.T @ Q_normal

print("Basis orthogonality check:")
print(np.round(basis_check, 5))

## 7. Projection-Based Anomaly Detection

The first two eigenvectors represent 95.66% of the normal
machine operating variation.

A new machine observation can be projected onto this
normal operating subspace.

If the observation is close to the normal subspace, the
machine is likely operating normally.

If it is far from the normal subspace, the large projection
residual indicates abnormal machine behavior.

The anomaly score is defined as the Euclidean norm of
the projection residual.

In [ ]:
# Project all machine observations onto the normal subspace

X_projection = (
    X_standardized @ Q_normal
) @ Q_normal.T

print("Projected data shape:", X_projection.shape)

In [ ]:
residuals = X_standardized - X_projection

print("Residual matrix shape:", residuals.shape)

In [ ]:
anomaly_scores = np.linalg.norm(
    residuals,
    axis=1
)

print("First 10 anomaly scores:")
print(anomaly_scores[:10])

In [ ]:
normal_scores = anomaly_scores[
    df["Fault_Type"] == "Normal"
]

print("Number of normal scores:", len(normal_scores))

print("\nNormal machine anomaly scores:")
print(normal_scores[:10])

In [ ]:
threshold = np.percentile(
    normal_scores,
    95
)

print("Anomaly detection threshold:", threshold)

In [ ]:
detected_status = np.where(
    anomaly_scores > threshold,
    "Anomaly",
    "Normal"
)

print("First 20 detected statuses:")
print(detected_status[:20])

In [ ]:
df["Anomaly_Score"] = anomaly_scores
df["Detected_Status"] = detected_status

print(df[
    ["Fault_Type", "Anomaly_Score", "Detected_Status"]
].head(20))

In [ ]:
print("Detection results:")
print(df["Detected_Status"].value_counts())

In [ ]:
print("Detection results by actual fault type:")
print(
    pd.crosstab(
        df["Fault_Type"],
        df["Detected_Status"]
    )
)

## 8. Improved PCA-Based Anomaly Detection

The previous detector used the residual from the normal operating
subspace. However, some faults may occur along the dominant
eigenvector directions and therefore may not produce a large
projection residual.

To improve detection, the normal machine data is used to learn
the mean and standard deviation.

The detector then considers both:

1. Deviation outside the normal subspace
2. Unusual movement along the dominant eigenvector directions

This provides a more complete measure of machine abnormality.

In [ ]:
normal_mean = X_normal.mean(axis=0)
normal_std = X_normal.std(axis=0)

X_normal_scaled = (
    X_normal - normal_mean
) / normal_std

X_all_scaled = (
    X - normal_mean
) / normal_std

print("Normal-based standardization completed.")
print("Normal data shape:", X_normal_scaled.shape)
print("All data shape:", X_all_scaled.shape)

In [ ]:
covariance_normal = np.cov(
    X_normal_scaled,
    rowvar=False
)

print("Normal-data covariance matrix:")
print(covariance_normal)

In [ ]:
eigenvalues_normal, eigenvectors_normal = np.linalg.eigh(
    covariance_normal
)

# Sort from largest to smallest
idx = np.argsort(eigenvalues_normal)[::-1]

eigenvalues_normal = eigenvalues_normal[idx]
eigenvectors_normal = eigenvectors_normal[:, idx]

print("Eigenvalues:")
print(eigenvalues_normal)

In [ ]:
explained_normal = (
    eigenvalues_normal /
    eigenvalues_normal.sum()
)

cumulative_normal = np.cumsum(
    explained_normal
)

for i in range(len(eigenvalues_normal)):
    print(
        "Eigenvector", i + 1,
        ":",
        round(explained_normal[i] * 100, 2),
        "%",
        "| Cumulative:",
        round(cumulative_normal[i] * 100, 2),
        "%"
    )

In [ ]:
k_improved = 2

Q_improved = eigenvectors_normal[:, :k_improved]

print("Improved normal subspace basis:")
print(Q_improved)

print("\nBasis shape:", Q_improved.shape)

In [ ]:
basis_check_improved = Q_improved.T @ Q_improved

print("Orthonormality check:")
print(np.round(basis_check_improved, 5))

In [ ]:
X_improved_projection = (
    X_all_scaled @ Q_improved
) @ Q_improved.T

print("Improved projected data shape:",
      X_improved_projection.shape)

In [ ]:
improved_residuals = (
    X_all_scaled - X_improved_projection
)

print("Improved residual matrix shape:",
      improved_residuals.shape)

In [ ]:
improved_anomaly_scores = np.linalg.norm(
    improved_residuals,
    axis=1
)

print("First 10 improved anomaly scores:")
print(improved_anomaly_scores[:10])

In [ ]:
normal_improved_scores = improved_anomaly_scores[
    df["Fault_Type"] == "Normal"
]

print("Number of normal scores:",
      len(normal_improved_scores))

In [ ]:
improved_threshold = np.percentile(
    normal_improved_scores,
    95
)

print(
    "Improved anomaly detection threshold:",
    improved_threshold
)

In [ ]:
improved_status = np.where(
    improved_anomaly_scores > improved_threshold,
    "Anomaly",
    "Normal"
)

df["Improved_Anomaly_Score"] = improved_anomaly_scores
df["Improved_Status"] = improved_status

print("Improved detection results:")
print(df["Improved_Status"].value_counts())

In [ ]:
print("Improved detection by actual fault type:")
print(
    pd.crosstab(
        df["Fault_Type"],
        df["Improved_Status"]
    )
)

## 9. Anomaly Score Visualization

The anomaly score measures the distance of a machine observation
from the learned normal operating subspace.

The 95th-percentile score of normal machines is used as the
anomaly detection threshold.

Observations above the threshold are classified as anomalous.

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    improved_anomaly_scores,
    label="Anomaly Score"
)

plt.axhline(
    improved_threshold,
    linestyle="--",
    label="Anomaly Threshold"
)

plt.xlabel("Machine Observation")
plt.ylabel("Anomaly Score")
plt.title("Machine Anomaly Detection")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    improved_anomaly_scores,
    label="Anomaly Score"
)

plt.axhline(
    improved_threshold,
    linestyle="--",
    label="Threshold"
)

# Highlight detected anomalies
anomaly_indices = np.where(
    improved_anomaly_scores > improved_threshold
)[0]

plt.scatter(
    anomaly_indices,
    improved_anomaly_scores[anomaly_indices],
    label="Detected Anomaly"
)

plt.xlabel("Machine Observation")
plt.ylabel("Anomaly Score")
plt.title("Linear Algebra-Based Machine Anomaly Detection")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
detection_table = pd.crosstab(
    df["Fault_Type"],
    df["Improved_Status"]
)

fault_detection_rate = (
    detection_table["Anomaly"]
    / detection_table.sum(axis=1)
    * 100
)

print("Fault detection rates:")
print(fault_detection_rate)

In [ ]:
plt.figure(figsize=(10, 6))

fault_detection_rate.plot(kind="bar")

plt.ylabel("Detection Rate (%)")
plt.xlabel("Fault Type")
plt.title("Fault Detection Performance")

plt.xticks(rotation=20)
plt.ylim(0, 110)

plt.tight_layout()
plt.show()

## 10. Sensor-Wise Fault Explanation

The anomaly score tells us whether a machine observation is abnormal,
but it does not explain which sensor contributes most to the anomaly.

The projection residual contains the deviation of each sensor from
the learned normal operating subspace.

The squared residual of each sensor is used to measure its contribution
to the overall anomaly.

In [ ]:
sensor_contribution = improved_residuals ** 2

print("Sensor contribution matrix shape:")
print(sensor_contribution.shape)

In [ ]:
sensor_names = [
    "Temperature",
    "Vibration",
    "Current",
    "Acoustic",
    "RPM"
]

dominant_sensor_index = np.argmax(
    sensor_contribution,
    axis=1
)

dominant_sensor = [
    sensor_names[i]
    for i in dominant_sensor_index
]

df["Dominant_Sensor"] = dominant_sensor

print(
    df[
        ["Fault_Type",
         "Improved_Anomaly_Score",
         "Improved_Status",
         "Dominant_Sensor"]
    ].head(20)
)

In [ ]:
fault_examples = df[
    df["Fault_Type"] != "Normal"
][
    [
        "Fault_Type",
        "Improved_Anomaly_Score",
        "Improved_Status",
        "Dominant_Sensor"
    ]
].head(20)

print(fault_examples)

In [ ]:
fault_sensor_summary = pd.crosstab(
    df["Fault_Type"],
    df["Dominant_Sensor"]
)

print("Dominant sensor by fault type:")
print(fault_sensor_summary)

## 11. Machine Health Score and Maintenance Recommendation

The anomaly score measures how far a machine observation deviates
from the learned normal operating subspace.

To make the result easier for an operator to understand, the
anomaly score is converted into a machine health score.

A higher health score indicates healthier machine operation.

In [ ]:
health_score = (
    100 * (1 - improved_anomaly_scores / improved_threshold)
)

# Keep the score between 0 and 100
health_score = np.clip(
    health_score,
    0,
    100
)

df["Health_Score"] = health_score

print(df[
    [
        "Fault_Type",
        "Improved_Anomaly_Score",
        "Health_Score",
        "Improved_Status"
    ]
].head(20))

In [ ]:
health_score = (
    100
    - 50 * (
        improved_anomaly_scores
        / improved_threshold
    )
)

health_score = np.clip(
    health_score,
    0,
    100
)

df["Health_Score"] = health_score

print(df[
    [
        "Fault_Type",
        "Improved_Anomaly_Score",
        "Health_Score",
        "Improved_Status"
    ]
].head(20))

In [ ]:
maintenance_status = []

for score in df["Improved_Anomaly_Score"]:
    if score <= improved_threshold:
        maintenance_status.append("Normal Operation")
    elif score <= improved_threshold * 1.5:
        maintenance_status.append("Monitor Machine")
    else:
        maintenance_status.append("Maintenance Recommended")

df["Maintenance_Status"] = maintenance_status

print("Maintenance_Status column created successfully!")

In [ ]:
# Final Machine Health Report

# Select the first detected faulty/anomalous machine
demo_index = df.index[
    (df["Fault_Type"] != "Normal") &
    (df["Improved_Status"] == "Anomaly")
][0]

machine = df.loc[demo_index]

print("=" * 50)
print("        MACHINE HEALTH REPORT")
print("=" * 50)

print("Sample ID:", machine["Sample_ID"])
print("Anomaly Score:", round(machine["Improved_Anomaly_Score"], 3))
print("Threshold:", round(improved_threshold, 3))
print("Status:", machine["Improved_Status"])
print("Dominant Sensor:", machine["Dominant_Sensor"])
print("Health Score:", round(machine["Health_Score"], 2))
print("Maintenance Status:", machine["Maintenance_Status"])

print("=" * 50)

In [ ]:
# Diagonalization of the covariance matrix

Lambda = np.diag(eigenvalues_normal)

reconstructed_covariance = (
    eigenvectors_normal
    @ Lambda
    @ eigenvectors_normal.T
)

reconstruction_error = np.max(
    np.abs(covariance_normal - reconstructed_covariance)
)

print("Original covariance matrix:")
print(covariance_normal)

print("\nReconstructed covariance matrix:")
print(reconstructed_covariance)

print("\nMaximum reconstruction error:",
      reconstruction_error)

In [ ]:
# LU Decomposition of the covariance matrix

from scipy.linalg import lu

P, L, U = lu(covariance_normal)

print("L matrix:")
print(L)

print("\nU matrix:")
print(U)

print("\nPermutation matrix P:")
print(P)

# Verify A = P @ L @ U
reconstructed_LU = P @ L @ U

lu_error = np.max(
    np.abs(covariance_normal - reconstructed_LU)
)

print("\nMaximum LU reconstruction error:", lu_error)

In [ ]:
# Rank-Nullity Theorem

number_of_features = len(features)

nullity = number_of_features - rank

print("Number of features:", number_of_features)
print("Rank:", rank)
print("Nullity:", nullity)

print("\nRank + Nullity =", rank + nullity)

In [ ]:
# Rank-Nullity Theorem

number_of_features = len(features)

nullity = number_of_features - rank

print("Number of features:", number_of_features)
print("Rank:", rank)
print("Nullity:", nullity)

print("\nRank + Nullity =", rank + nullity)

In [ ]:
# Final Detection Results

results = pd.DataFrame({
    "Fault Type": df["Fault_Type"],
    "Anomaly Score": df["Improved_Anomaly_Score"],
    "Status": df["Improved_Status"],
    "Dominant Sensor": df["Dominant_Sensor"],
    "Health Score": df["Health_Score"],
    "Maintenance": df["Maintenance_Status"]
})

print(results.head(10))

In [ ]:
# Fault Detection Performance

performance = pd.crosstab(
    df["Fault_Type"],
    df["Improved_Status"]
)

performance["Detection_Rate_%"] = (
    performance["Anomaly"] /
    performance.sum(axis=1)
) * 100

print(performance)

In [ ]:
# Interactive Machine Checker

machine_id = int(input("Enter Machine ID: "))

machine_rows = df[df["Sample_ID"] == machine_id]

if len(machine_rows) == 0:
    print("Machine ID not found.")
else:
    machine = machine_rows.iloc[0]

    print("\nMachine found!")
    print("Machine ID:", machine["Sample_ID"])
    print("Temperature:", machine["Temperature_C"], "°C")
    print("Vibration:", machine["Vibration_RMS_g"], "g")
    print("Current:", machine["Current_RMS_A"], "A")
    print("Acoustic:", machine["Acoustic_RMS_Pa"], "Pa")
    print("RPM:", machine["RPM"])

In [ ]:
# Complete Machine Analysis

machine_id = int(input("Enter Machine ID: "))

machine_rows = df[df["Sample_ID"] == machine_id]

if len(machine_rows) == 0:
    print("Machine ID not found.")

else:
    machine = machine_rows.iloc[0]

    # Get the five sensor values
    machine_values = machine[features].to_numpy(dtype=float)

    # Standardize using normal-machine baseline
    machine_scaled = (machine_values - normal_mean) / normal_std

    # Project onto normal operating subspace
    machine_projection = (
        machine_scaled @ Q_improved
    ) @ Q_improved.T

    # Calculate residual
    machine_residual = machine_scaled - machine_projection

    # Calculate anomaly score
    machine_anomaly_score = np.linalg.norm(machine_residual)

    # Determine status
    if machine_anomaly_score > improved_threshold:
        machine_status = "ANOMALY"
    else:
        machine_status = "NORMAL"

    # Health score
    machine_health_score = 100 - 50 * (
        machine_anomaly_score / improved_threshold
    )

    machine_health_score = np.clip(
        machine_health_score, 0, 100
    )

    # Find dominant sensor
    sensor_contributions = machine_residual ** 2
    dominant_index = np.argmax(sensor_contributions)
    dominant_sensor = sensor_names[dominant_index]

    # Maintenance recommendation
    if machine_anomaly_score <= improved_threshold:
        maintenance = "Normal Operation"
    elif machine_anomaly_score <= improved_threshold * 1.5:
        maintenance = "Monitor Machine"
    else:
        maintenance = "Maintenance Recommended"

    # Display report
    print("\n" + "=" * 55)
    print("             MACHINE HEALTH REPORT")
    print("=" * 55)

    print("\nMachine ID:", machine["Sample_ID"])

    print("\n--- Sensor Readings ---")
    print("Temperature :", machine["Temperature_C"], "°C")
    print("Vibration   :", machine["Vibration_RMS_g"], "g")
    print("Current     :", machine["Current_RMS_A"], "A")
    print("Acoustic    :", machine["Acoustic_RMS_Pa"], "Pa")
    print("RPM         :", machine["RPM"])

    print("\n--- Mathematical Analysis ---")
    print("Anomaly Score :", round(machine_anomaly_score, 3))
    print("Threshold     :", round(improved_threshold, 3))
    print("Status        :", machine_status)
    print("Health Score  :", round(machine_health_score, 2), "/ 100")
    print("Dominant Sensor:", dominant_sensor)

    print("\n--- Maintenance Recommendation ---")
    print(maintenance)

    print("=" * 55)

In [ ]:
# Standardized Machine Sensor Performance

if len(machine_rows) != 0:

    standardized_values = machine_scaled

    sensor_labels = [
        "Temperature",
        "Vibration",
        "Current",
        "Acoustic",
        "RPM"
    ]

    plt.figure(figsize=(10, 6))

    plt.bar(sensor_labels, standardized_values)

    plt.axhline(0, linewidth=1)

    plt.title("Standardized Sensor Performance")
    plt.xlabel("Sensor")
    plt.ylabel("Standardized Value (Z-score)")

    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# Anomaly Score vs Normal Threshold

plt.figure(figsize=(8, 6))

plt.bar(
    ["Machine Anomaly Score", "Normal Threshold"],
    [machine_anomaly_score, improved_threshold]
)

plt.ylabel("Score")
plt.title("Machine Anomaly Score vs Normal Threshold")

plt.tight_layout()
plt.show()

In [ ]:
# Sensor Contribution to Anomaly

contribution_values = machine_residual ** 2

sensor_labels = [
    "Temperature",
    "Vibration",
    "Current",
    "Acoustic",
    "RPM"
]

plt.figure(figsize=(10, 6))

plt.bar(sensor_labels, contribution_values)

plt.title("Sensor Contribution to Anomaly")
plt.xlabel("Sensor")
plt.ylabel("Squared Residual")

plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
def check_machine(machine_id):

    machine_rows = df[df["Sample_ID"] == machine_id]

    if len(machine_rows) == 0:
        print("Machine ID not found.")
        return

    machine = machine_rows.iloc[0]

    # Sensor values
    machine_values = machine[features].to_numpy(dtype=float)

    # Standardization
    machine_scaled = (machine_values - normal_mean) / normal_std

    # Projection onto normal operating subspace
    machine_projection = (
        machine_scaled @ Q_improved
    ) @ Q_improved.T

    # Residual
    machine_residual = machine_scaled - machine_projection

    # Anomaly score
    anomaly_score = np.linalg.norm(machine_residual)

    # Status
    if anomaly_score > improved_threshold:
        status = "ANOMALY"
    else:
        status = "NORMAL"

    # Health score
    health_score = 100 - 50 * (
        anomaly_score / improved_threshold
    )

    health_score = np.clip(health_score, 0, 100)

    # Sensor contribution
    contributions = machine_residual ** 2

    dominant_index = np.argmax(contributions)
    dominant_sensor = sensor_names[dominant_index]

    # Maintenance recommendation
    if anomaly_score <= improved_threshold:
        maintenance = "Normal Operation"
    elif anomaly_score <= improved_threshold * 1.5:
        maintenance = "Monitor Machine"
    else:
        maintenance = "Maintenance Recommended"

    # -----------------------------
    # REPORT
    # -----------------------------

    print("\n" + "=" * 55)
    print("              MACHINE HEALTH REPORT")
    print("=" * 55)

    print("\nMachine ID:", machine["Sample_ID"])

    print("\n--- Sensor Readings ---")
    print("Temperature :", machine["Temperature_C"], "°C")
    print("Vibration   :", machine["Vibration_RMS_g"], "g")
    print("Current     :", machine["Current_RMS_A"], "A")
    print("Acoustic    :", machine["Acoustic_RMS_Pa"], "Pa")
    print("RPM         :", machine["RPM"])

    print("\n--- Analysis ---")
    print("Anomaly Score :", round(anomaly_score, 3))
    print("Threshold     :", round(improved_threshold, 3))
    print("Status        :", status)
    print("Health Score  :", round(health_score, 2), "/ 100")
    print("Dominant Sensor:", dominant_sensor)

    print("\n--- Recommendation ---")
    print(maintenance)

    print("=" * 55)

    # -----------------------------
    # GRAPH 1: STANDARDIZED SENSORS
    # -----------------------------

    plt.figure(figsize=(9, 5))

    plt.bar(sensor_names, machine_scaled)

    plt.axhline(0, linewidth=1)

    plt.title("Standardized Sensor Performance")
    plt.ylabel("Standardized Value (Z-score)")
    plt.xticks(rotation=20)

    plt.tight_layout()
    plt.show()

    # -----------------------------
    # GRAPH 2: ANOMALY VS THRESHOLD
    # -----------------------------

    plt.figure(figsize=(7, 5))

    plt.bar(
        ["Machine Score", "Normal Threshold"],
        [anomaly_score, improved_threshold]
    )

    plt.ylabel("Score")
    plt.title("Anomaly Score vs Normal Threshold")

    plt.tight_layout()
    plt.show()

    # -----------------------------
    # GRAPH 3: SENSOR CONTRIBUTION
    # -----------------------------

    plt.figure(figsize=(9, 5))

    plt.bar(sensor_names, contributions)

    plt.title("Sensor Contribution to Anomaly")
    plt.ylabel("Squared Residual")
    plt.xticks(rotation=20)

    plt.tight_layout()
    plt.show()

In [ ]:
def check_machine(machine_id):

    machine_rows = df[df["Sample_ID"] == machine_id]

    if len(machine_rows) == 0:
        print("Machine ID not found.")
        return

    machine = machine_rows.iloc[0]

    # Get sensor values
    machine_values = machine[features].to_numpy(dtype=float)

    # Standardize using normal-machine statistics
    machine_scaled = (machine_values - normal_mean) / normal_std

    # Project onto normal operating subspace
    machine_projection = (machine_scaled @ Q_improved) @ Q_improved.T

    # Calculate residual
    machine_residual = machine_scaled - machine_projection

    # Calculate anomaly score
    anomaly_score = np.linalg.norm(machine_residual)

    # Determine status
    if anomaly_score > improved_threshold:
        status = "ANOMALY"
    else:
        status = "NORMAL"

    # Health score
    health_score = 100 - 50 * (anomaly_score / improved_threshold)
    health_score = np.clip(health_score, 0, 100)

    # Sensor contribution
    contributions = machine_residual ** 2
    dominant_index = np.argmax(contributions)
    dominant_sensor = sensor_names[dominant_index]

    # Maintenance recommendation
    if anomaly_score <= improved_threshold:
        maintenance = "Normal Operation"
    elif anomaly_score <= improved_threshold * 1.5:
        maintenance = "Monitor Machine"
    else:
        maintenance = "Maintenance Recommended"

    # Print report
    print("=" * 55)
    print("              MACHINE HEALTH REPORT")
    print("=" * 55)

    print("Machine ID:", machine_id)

    print("\n--- Sensor Readings ---")
    print(f"Temperature : {machine['Temperature_C']:.3f} °C")
    print(f"Vibration   : {machine['Vibration_RMS_g']:.5f} g")
    print(f"Current     : {machine['Current_RMS_A']:.4f} A")
    print(f"Acoustic    : {machine['Acoustic_RMS_Pa']:.6f} Pa")
    print(f"RPM         : {machine['RPM']:.2f}")

    print("\n--- Analysis ---")
    print(f"Anomaly Score : {anomaly_score:.3f}")
    print(f"Threshold     : {improved_threshold:.3f}")
    print(f"Status        : {status}")
    print(f"Health Score  : {health_score:.2f} / 100")
    print(f"Dominant Sensor: {dominant_sensor}")

    print("\n--- Recommendation ---")
    print(maintenance)

    print("=" * 55)

In [ ]:
machine_id = int(input("Enter Machine ID: "))
check_machine(machine_id)

In [ ]:
import pandas as pd
import numpy as np

def check_machine():
    # Load the latest CSV whenever the checker runs
    df = pd.read_csv("machine_fault_project_dataset_5_parameters_random_ids.csv")

    machine_id = int(input("Enter Machine ID: "))

    machine_rows = df[df["Sample_ID"] == machine_id]

    if machine_rows.empty:
        print("Machine ID not found in the CSV.")
        print("Check that you saved the CSV and entered the correct ID.")
        return

    machine = machine_rows.iloc[0]

    # Get the five sensor readings
    sensor_values = machine[features].to_numpy(dtype=float)

    # Use the normal baseline already calculated by your model
    scaled_values = (sensor_values - normal_mean) / normal_std

    # Project onto the learned normal operating subspace
    projected_values = (scaled_values @ Q_improved) @ Q_improved.T
    residual = scaled_values - projected_values

    # Calculate anomaly score
    anomaly_score = np.linalg.norm(residual)
    status = "ANOMALY" if anomaly_score > improved_threshold else "NORMAL"

    # Project-defined health score
    health_score = 100 / (1 + anomaly_score / improved_threshold)

    # Identify the sensor contributing most to the residual
    contributions = residual ** 2
    sensor_names = [
        "Temperature", "Vibration", "Current", "Acoustic", "RPM"
    ]
    dominant_sensor = sensor_names[np.argmax(contributions)]

    # Prototype maintenance rule
    if anomaly_score <= improved_threshold:
        recommendation = "Normal Operation"
    elif anomaly_score <= improved_threshold * 1.5:
        recommendation = "Monitor Machine"
    else:
        recommendation = "Maintenance Recommended"

    # Display report
    print("\n" + "=" * 50)
    print("             MACHINE HEALTH REPORT")
    print("=" * 50)
    print("Machine ID:", machine["Sample_ID"])
    print("Dataset Fault Label:", machine["Fault_Type"])
    print("\n--- Sensor Readings ---")
    print(f"Temperature : {machine['Temperature_C']} °C")
    print(f"Vibration   : {machine['Vibration_RMS_g']} g")
    print(f"Current     : {machine['Current_RMS_A']} A")
    print(f"Acoustic    : {machine['Acoustic_RMS_Pa']} Pa")
    print(f"RPM         : {machine['RPM']}")

    print("\n--- Model Analysis ---")
    print(f"Anomaly Score : {anomaly_score:.3f}")
    print(f"Threshold     : {improved_threshold:.3f}")
    print("Model Status  :", status)
    print(f"Health Score  : {health_score:.2f} / 100")
    print("Dominant Sensor:", dominant_sensor)

    print("\n--- Recommendation ---")
    print(recommendation)
    print("=" * 50)

# Run the checker
check_machine()